In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os


In [ ]:
!pip install faster-whisper

In [ ]:
import os
import time
import re
import csv
import queue
import threading
import torchaudio
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from faster_whisper import WhisperModel, BatchedInferencePipeline

In [ ]:
AUDIO_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio"
OUTPUT_CSV = "/kaggle/working/submission_faster_whisper.csv"
MODEL_NAME = "teamvillagers/whisper-bengali-ct2"

COMPUTE_TYPE = "float16"
BATCH_SIZE = 32
CHUNK_LENGTH = 20
BEAM_SIZE = 5

In [ ]:
BN_PUNCT = r"[।,!?;:“”\"'()\[\]{}—–…]"

def clean_text(text: str) -> str:

    text = text.replace("প্রশংসা", "")


    text = re.sub(BN_PUNCT, "", text)
    text = text.replace("?", "").replace("\n", " ")


    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ======================
# MULTI-GPU SETUP
# ======================
audio_files = sorted(Path(AUDIO_DIR).glob("*.wav"))
print(f"Queueing {len(audio_files)} files for 2 GPUs...")


file_queue = queue.Queue()
for f in audio_files:
    file_queue.put(f)


pbar = tqdm(total=len(audio_files), desc="Transcribing (2x T4)", unit="file", dynamic_ncols=True)
pbar_lock = threading.Lock()

def worker(gpu_id):
    """Worker thread that processes files on a specific GPU."""
    # Kaggle T4x2 has 4 vCPUs total. We restrict each model to 2 CPU threads to prevent CPU thrashing.
    model = WhisperModel(
        MODEL_NAME,
        device="cuda",
        device_index=gpu_id,
        compute_type=COMPUTE_TYPE,
        cpu_threads=2
    )
    pipeline = BatchedInferencePipeline(model=model)

    local_results = []
    local_total_sec = 0.0

    while True:
        try:

            audio_path = file_queue.get_nowait()
        except queue.Empty:
            break

        info = torchaudio.info(str(audio_path))
        local_total_sec += info.num_frames / info.sample_rate

        segments, _ = pipeline.transcribe(
            str(audio_path),
            language="bn",
            beam_size=BEAM_SIZE,
            chunk_length=CHUNK_LENGTH,
            batch_size=BATCH_SIZE,
            condition_on_previous_text=False,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=1000, speech_pad_ms=2000)
        )


        raw_text = " ".join(seg.text for seg in segments)
        final_text = clean_text(raw_text)

        local_results.append((audio_path.stem, final_text))


        with pbar_lock:
            pbar.update(1)

    return local_results, local_total_sec

infer_start = time.time()

# Launch exactly 2 threads, one for cuda:0 and one for cuda:1
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(worker, i) for i in range(2)]

# Collect results from both GPUs
final_results = []
total_audio_sec = 0.0

for f in futures:
    res, sec = f.result()
    final_results.extend(res)
    total_audio_sec += sec

pbar.close()
infer_time = time.time() - infer_start

# ======================
# WRITE CSV & TIMING
# ======================
# Sort results by filename to ensure CSV order is consistent
final_results.sort(key=lambda x: x[0])

with open(OUTPUT_CSV, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "transcript"])
    writer.writerows(final_results)

rtf = infer_time / total_audio_sec

print("\n================ TIMING =======================")
print(f"Total audio duration : {total_audio_sec:.2f} sec")
print(f"Inference time       : {infer_time:.2f} sec")
print(f"Real-Time Factor     : {rtf:.4f} (lower is better)")
print(f"✅ CSV saved to: {OUTPUT_CSV}")